# Лабораторная работа 4. Введение в машинное обучение: knn, деревянные алгоритмы и ансамблирование. Анализ и сравнение.

![](https://newapplift-production.s3.amazonaws.com/comfy/cms/files/files/000/001/201/original/machine-learning-robots-dilbert.gif)

Результат лабораторной работы − отчет. Мы предпочитаем принимать отчеты в формате ноутбуков Jupyter (ipynb-файл). Постарайтесь сделать ваш отчет интересным рассказом, последовательно отвечающим на вопросы из заданий. Помимо ответов на вопросы, в отчете так же должен быть код, однако чем меньше кода, тем лучше всем: нам − меньше проверять, вам —  проще найти ошибку или дополнить эксперимент. При проверке оценивается четкость ответов на вопросы, аккуратность отчета и кода.


### Оценивание и штрафы
* Не копируйте классы между заданиями, объявите решающие модели один раз, а затем их инстанциируйте в каждой из ячеек
* Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи).
* Максимально допустимая оценка за работу — 22 балла. При выставлении оценки будут учитываться только 14 балла за эту лабораторную, 8 баллов считаются дополнительными.
* Сдавать задание после указанного срока сдачи нельзя.
* Не оцениваются задания с удалёнными формулировками.
* Не оценивается лабораторная работа целиком, если она была выложена в открытый источник.


## Метрика качества

Обучение и оценка качества модели производятся на независимых множествах примеров. Как правило, имеющующиеся примеры разбивают на два подмножества: обучающее (train) и тестовое (test). Выбор пропорции разбиения — компромисс. Действительно, большой размер обучения ведет к более качественным алгоритмам, но бОльшему шуму при оценке модели на тесте. И наоборот, большой размер тестовой выборки ведет к менее шумной оценке качества, однако обученные модели получаются менее точными.

Многие модели классификации предсказывают оценку принадлежности положительному классу $\tilde{y}(x) \in R$ (например, вероятность принадлежности классу 1). После этого принимают решение о классе объекта путем сравнения оценки с некоторым порогом $\theta$:

$$y(x) = 
\begin{cases}
+1, &\text{если} \; \tilde{y}(x) \geq \theta \\
-1, &\text{если} \; \tilde{y}(x) < \theta
\end{cases}
$$

В этом случае можно рассматривать метрики, которые умеют работать с исходным ответом классификатора. В задании мы будем работать с метрикой AUC-ROC, которую в данном случае можно считать как долю неправильно упорядоченных пар объектов, отсортированных по возрастанию предсказанной оценки принадлежности классу 1 (более подробно можно узнать на следующих лекциях или, например, [здесь](https://github.com/esokolov/ml-course-msu/blob/master/ML15/lecture-notes/Sem05_metrics.pdf)). Детального понимания принципов работы метрики AUC-ROC для выполнения этой лабораторной не требуется. В sklearn данная метрика реализуется [следующей функцией](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html).

# Подбор гиперпараметров и сравнение моделей (14-22 баллов)

В этом задании также можно получить дополнительные баллы за использование ваших реализаций алгоритмов при сравнении: knn, decision tree, random forest, mlp.

Баллы будут засчитываться, только если вы при решении заданий будете использовать только вашу реализацию алгоритма или алгоритмов и решите как минимум 4 задания.

Дополнительные баллы следующие:
1. knn (с предыдущей лабы):
   1. plain knn - 1 балл
   2. ann - 2 балла
2. decision tree - 2 балла
3. random forest:
   1. random forest на деревьях из sklearn - 1 балл
   2. random forest на собственной реализации деревьев - 2 балла
4. MLP (с предыдущей лабы):
   1. Своя реализация MLP - 2 балла

Перед решением данного задания, опишите в окошке снизу, какие ваши реализации вы будете использовать, чтобы я их не пропустил случайно:

In [16]:
# TODO: опишите, какие свои реализации вы будете использовать


### Подбор гиперпараметров модели

В задачах машинного обучения следует различать параметры модели и гиперпараметры (структурные параметры). Обычно параметры модели настраиваются в ходе обучения (например, веса в линейной модели или структура решающего дерева), в то время как гиперпараметры задаются заранее (например, значение силы регуляризации в линейной модели или максимальная глубина решающего дерева). Каждая модель, как правило, имеет множество гиперпараметров и нет универсальных наборов гиперпараметров, оптимально работающих во всех задачах, поэтому для каждой задачи нужно подбирать свой набор.

Для оптимизации гиперпараметров модели часто используют _перебор по сетке (grid search)_: для каждого гиперпараметра выбирается несколько значений, далее перебираются все комбинации значений и выбирается комбинация, на которой модель показывает лучшее качество (с точки зрения оптимизируемой метрики). Однако, в этом случае нужно грамотно оценивать построенную модель, а именно, делать разбиение на обучающую и тестовую выборку. Есть несколько схем, как это можно реализовать: 

 - Разбить имеющуюся выборку на обучающую и тестовую. В этом случае сравнение большого числа моделей при переборе гиперпараметров приводит к ситуации, когда лучшая на тестовой подвыборке модель не сохраняет свои качества на новых данных. Можно сказать, что происходит _переобучение_ на тестовую выборку.
 - Для устранения описанной выше проблемы можно разбить данные на 3 непересекающихся подвыборки: обучение, валидация и тест. Валидационную подвыборку используют для сравнения моделей, а тестовую — для окончательной оценки качества и сравнения семейств моделей с подобранными гиперпараметрами.
 - Другой способ сравнения моделей **рекомендуемый**— [кросс-валидация](http://bit.ly/1CHXsNH) . Существуют различные схемы кросс-валидации:
  - Leave-One-Out
  - K-Fold
  - Многократное случайное разбиение выборки
  - Бутстрап выборки
  
Кросс-валидация вычислительно затратна, особенно если вы делаете перебор по сетке с очень большим числом комбинаций. С учетом конечности времени на выполнение задания, возникает ряд компромиссов: 
  - сетку гиперпараметров можно делать более разреженной, перебирая меньше значений каждого гиперпараметра; однако не стоит забывать, что в таком случае можно пропустить хорошую комбинацию гиперпараметров;
  - кросс-валидацию можно делать с меньшим числом разбиений или фолдов, но в таком случае оценка качества становится более шумной и увеличивается риск выбрать неоптимальный набор гиперпараметров из-за случайности разбиения;
  - гиперпараметры можно оптимизировать последовательно (жадно) — один за другим, а не перебирать все комбинации; такая стратегия не всегда приводит к оптимальному набору;
  - перебирать не все комбинации гиперпараметров, а небольшое число каким-то образом выбранных.

### Задание

В этой части лабораторной работы мы научимся обучать модели машинного обучения, корректно ставить эксперименты, подбирать гиперпараметры, сравнивать и смешивать модели. Вам предлагается решить задачу бинарной классификации, а именно, построить алгоритм, определяющий, превысит ли средний заработок человека порог $50k.

Ссылка на датасет с его описанием [здесь](http://archive.ics.uci.edu/dataset/2/adult).

Более подробно про признаки можно прочитать в файле *adult.names*. Целевой признак записан в переменной *>50K,<=50K*.

Загрузите набор данных *data.adult.csv*. Чтобы лучше понимать, с чем вы работаете/корректно ли вы загрузили данные, можно вывести несколько первых строк на экран.

Иногда в данных встречаются пропуски. Способ обозначения пропусков либо прописывается в описании к данным, либо на месте пропуска после чтения данных оказывается значение [NaN](https://numpy.org/doc/stable/user/misc.html). Более подробно о работе с пропусками в Pandas можно прочитать, например, [здесь](http://pandas.pydata.org/pandas-docs/stable/missing_data.html). 

В данном датасете пропущенные значения обозначены как "?". Базовую обработку можно взять из первой части лабораторной работы.

**Задание 1 (0.5 балла).** Обычно после загрузки датасета всегда необходима его некоторая предобработка. В данном случае она будет заключаться в следующем: 
 - Найдите все признаки, имеющие пропущенные значения. Замените их по совему усмотрению - или константой, или по определенной стратегии при помощи [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) замените на среднее, медиану или моду.
 - Сохраните целевую переменную (ту, которую мы хотим предсказывать) в отдельную переменную, удалите ее из датасета и преобразуйте к бинарному формату.
 - Обратите внимание, что не все признаки являются вещественными (числовыми). В начале мы будем работать только с вещественными признаками. Сохраните их отдельно.
 - Давайте учиться оформлять предобработку датасета по принятым стандартам. Используйте [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) для оформления решения (предобработка + модель) и  [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) для преобразования столбцов. Вы увидите, что это очень удобно, так как позволяет тюнить одновременно и модель, и предобработку (например, стратегию запорлнения пропусков).

### 1 Обучение классификаторов на вещественных признаках (6.5 баллов)

В данном разделе необходимо работать только с вещественными признаками и целевой переменной.

В начале посмотрим, как работает подбор гиперпараметров по сетке и как влияет на качество разбиение выборки. Сейчас и далее будем рассматривать 4 алгоритма:
 - [kNN](http://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
 - [DecisonTree](http://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier)
 - [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
 - [MLP](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html#sklearn.neural_network.MLPClassifier)

Для начала  для первых 4 алгоритмов выберите один гиперпараметр, который будем оптимизировать:
 - kNN — число соседей (*n_neighbors*);
 - DecisonTree — глубина дерева (*max_depth*);
 - Random Forest - глубина дерева (*max_depth*);
 - количество семплов для бутстрапа (*max_samples*), количество фичей для построения дерева (*max_features*);
 - MLP - количество слоев и глубина слоев (*hidden_layer_sizes*) (тут много не перебирайте);


Значения остальных гиперпараметров оставляйте по умолчанию. Для подбора гиперпараметров воспользуйтесь перебором по сетке, который реализован в классе [GridSearchCV](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV). В качестве схемы кросс-валидации используйте 5-Fold CV, которую можно задать с помощью класса [KFoldCV](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html#sklearn.model_selection.KFold).

![](https://i.stack.imgur.com/YWgro.gif)



**Задание 2 (2 балл).** Для каждого из первых 4 алгоритмов подберите оптимальные значения указанных гиперпараметров. Постройте график среднего значения качества по кросс-валидации алгоритма при заданном значении гиперпараметра, на котором также отобразите доверительный интервал.

Для получения значения качества на каждом фолде, среднего значение качества и другой полезной информации можно воспользоваться полем [*cv results_*](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV)

У какого алгоритма наибольшее среднее значение качества? Наименьший доверительный интервал?

**Задание 3 (1 балл).** Теперь подберём число деревьев (*n_estimators*) в алгоритме RandomForest. Как известно, в общем случае Random Forest не переобучается с увеличением количества деревьев. Подберите количество деревьев, начиная с которого качество на кросс-валидации стабилизируется. Обратите внимание, что для проведения этого эксперимента не нужно с нуля обучать много случайных лесов с разным количеством деревьев: обучите один случайный лес с максимальным интересным количеством деревьев, а затем рассмотрите подмножества деревьев разных размеров, состоящие из деревьев построенного леса (поле [*estimators_*](http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)). В дальнейших экспериментах используйте найденное количество деревьев.

Применить класс *GridSearchCV* в данном задании затруднительно, поэтому предлагается самостоятельно написать цикл по числу деревьев.

При обучении алгоритмов стоит обращать внимание не только на их качество, но и каким образом они работают с данными. В этой задаче получилось так, что некоторые из используемых алгоритмов чувствительны к масштабу признаков. Чтобы убедиться, что это могло повлиять на качество, давайте посмотрим на значения самих признаков.

**Задание 4 (0.5 балла).** Посмотрите на значения признаков *age*, *fnlwgt*, *capital-gain*. В чем заключается особенность данных? На какие из рассматриваемых алгоритмов это может повлиять? Может ли масштабирование повлиять на работу этих алгоритмов?

Масштабирование признаков можно выполнить, например, одним из следующих способов:
 - $x_{new} = \dfrac{x - \mu}{\sigma}$, где $\mu, \sigma$ — среднее и стандартное отклонение значения признака по всей выборке (см. функцию [scale](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.scale.html))
 - $x_{new} = \dfrac{x - x_{min}}{x_{max} - x_{min}}$, где $[x_{min}, x_{max}]$ — минимальный интервал значений признака

Похожие схемы масштабирования приведены в классах [StandardScaler](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html#sklearn.preprocessing.StandardScaler) и [MinMaxScaler](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html#sklearn.preprocessing.MinMaxScaler), причем **строго рекомендуется** использовать именно их, так как во избежание переобучения коэффициенты сдвига и масштаба необходимо считать не по всей выборке, а только по обучающей выборке, то есть применить метод *fit_transform* для обучающей выборки и метод *transform* для тестовой.
 
**Задание 5 (1 балл).** Подберите по кросс-валидации тот вариант масштабирования, который даёт лучшее качество - для каждого типа модели это может быть свой вариант масштабирования, в том числе отсуствие масштабирования тоже считается вариантом). Отмасштабируйте вещественные признаки и подберите оптимальные значения гиперпараметров аналогично пункту выше.

Изменилось ли качество некоторых алгоритмов и почему?

**Задание 6 (2 балла).** Теперь сделайте перебор нескольких гиперпараметров по сетке или с помощью оптуны и найдите оптимальные комбинации (лучшее среднее значение качества) для каждого алгоритма в данном случае: 
 - KNN — число соседей (*n_neighbors*) и метрика (*metric*);
 - DecisonTree — глубина дерева (*max_depth*) и минимальное число объектов в листе (*min_samples_leaf*);
 - RandomForest — минимальное число объектов в листе (*min_samples_leaf*) и максимальное число рассматриваемых признаков (*max_features*),  используйте найденное ранее количество деревьев.
 - MLP - структура классификатора (*hidden_layer_sizes*) и скорость обучения (*learning_rate* и *learning_rate_init*)
 
Обратите внимание, что эта операция может быть ресурсоемкой.

Какой из алгоритмов имеет наилучшее качество? 

**Задание 7 (1 балл).** Постройте для разных алгоритмов графики [кривых обучения](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.learning_curve.html), изображающие зависимость качества на тестовой и обучающей выборках от количества объектов, на которых обучаются модели. Посмотрите на поведение кривых и ответьте на вопросы:
* Может ли с ростом числа объектов убывать качество на тестовой выборке? А на обучающей? Почему?
* Для каких целей можно использовать знание качества на обучающей части выборки?
* Какой из алгоритмов лучше обучается на меньшем числе объектов?
* Может ли добавление новых объектов значительно повысить качество какого-то из алгоритмов или при существующем наборе данных для всех алгоритмов произошло насыщение?

### 2 Добавление категориальных признаков в модели (2 балла)

Пока мы не использовали нечисловые признаки, которые есть в датасете. Давайте посмотрим, правильно ли мы сделали, и увеличится ли качество моделей после добавлениях этих признаков. 

**Задание 8 (0.5 балла).** Преобразуйте все категориальные признаки с помощью метода one-hot-encoding [OneHotEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) или используйте встроенные возможности алгоритмов для работы с категориальными фичами.

Так как после кодирования признаков получилось достаточно много, в этой работе мы не будем подбирать заново оптимальные гиперпараметры для моделей с учетом новых признаков (хотя правильнее было бы это сделать). 

**Задание 9 (1.5 балла).** Добавьте к масштабированным вещественным признакам закодированные категориальные и обучите алгоритмы с наилучшими гиперпараметрами, найденными ранее. Дало ли добавление новых признаков прирост качества? Измеряйте качество, как и раньше, используя 5-Fold CV. Для этого удобно воспользоваться функцией [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html).

Отличается ли теперь наилучший классификатор от наилучшего в предыдущем пункте?

### 3 Смешивание моделей - blending (3 балла)

Во всех предыдущих пунктах мы получили много моделей, которые могут быть достаточно разными по своей природе. Часто на практике оказывается возможным увеличить качество предсказания путем смешивания разных моделей. Давайте посмотрим, действительно ли такой подход дает прирост в качестве.

Выберите из построенных моделей двух предыдущих пунктов две, которые дали наибольшее начество на кросс-валидации (обозначим их $clf_1$ и $clf_2$). Далее постройте новый классификатор, ответ которого на некотором объекте $x$ будет выглядеть следующим образом:

$$result(x) = clf_1(x) * \alpha + clf_2(x) * (1 - \alpha)$$

где $\alpha$ — гиперпараметр нового классификатора, $0\leqslant\alpha\leqslant1$.

**Задание 10 (1.5 балла).**
При реализации своих моделей хорошей практикой является создание sklearn-совместимых классов. Во-первых, такая реализация будет иметь стандартный интерфейс и позволит другим людям безболезненно обучать реализованные вами модели. Во-вторых, появляется возможность использовать любой функционал пакета sklearn, принимающий на вход модель, например, класс *GridSearchCV*, *learning_curve* и другие.

Создайте классификатор, который инициализируется двумя произвольными классификаторами и параметром $\alpha$. Во время обучения такой классификатор должен обучать обе базовые модели, а на этапе предсказания замешивать предсказания базовых моделей по формуле, указанной выше. 

Для создания пользовательского классификатора необходимо отнаследоваться от базовых классов [BaseEstimator](http://scikit-learn.org/stable/modules/generated/sklearn.base.BaseEstimator.html), [ClassifierMixin](http://scikit-learn.org/stable/modules/generated/sklearn.base.ClassifierMixin.html) и реализовать методы *\_\_init\_\_, fit, predict и predict_proba*. Пример sklearn-совместимого классификатора с комментариями можно найти [здесь](https://scikit-learn.org/stable/developers/develop.html#api-overview).

Данное задание можно будет зачесть, только если ваш код будет запускаться в следующем задании.

**Задание 11 (1.5 балла).** Теперь выберите два лучших построенных классификатора и обучите стекинг: подберите по сетке от 0 до 1 значение $\alpha$ для этого классификатора. Если класс реализован правильно, то вы cможете использовать *GridSearchCV*, как в случае с обычными классификаторами.
Изобразите на графике среднее качество по фолдам и доверительный интервал в зависимости от $\alpha$.

Дал ли этот подход прирост к качеству по сравнению с моделями, обученными по-отдельности? Поясните, почему даже простой блендинг моделей может влиять на итоговое качество?

Если RandomForest входит в топ-2 ваших алгоритмов, постройте дополнительно стекинг на двух лучших классификаторах без RandomForest, также подберите $\alpha$ и сравните полученный алгоритм с исходными алгоритмами и RandomForest: интересно сравнить два ансамбля.


### 4 Сравнение построенных моделей (1 балл)

![](http://cdn.shopify.com/s/files/1/0870/1066/files/compare_e8b89647-3cb6-4871-a976-2e36e5987773.png?1750043340268621065)

После того, как было построено много моделей, правильным продолжением является сравнение их между собой. Воспользуемся "ящиком с усами" (диаграммой размаха) для сравнения алгоритмов между собой (воспользуйтесь реализацией *box plot* в [seaborn](https://seaborn.pydata.org/generated/seaborn.boxplot.html) или [matplotlib](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.boxplot.html)). 


**Задание 12 (1 балл).** Для каждого типа классификатора (kNN, DecisionTree, Random Forest, MLP), а так же смешанной модели, выберите тот, который давал наилучшее качество на кросс-валидации, и постройте диаграмму размаха. Все классификаторы должны быть изображены на одном графике.
 
Сделайте общие итоговые выводы о классификаторах с точки зрения их работы с признаками и сложности самой модели (какие гиперпараметры есть у модели, сильно ли изменение значения гиперпараметра влияет на качество модели).